## DMEPOS-Supplier_Service Data Carpentry
The purpose of this notebook is to clean the DMEPOS-by Supplier and Service dataset for exploration and ML modeling.

The cleaning steps involved:
- Loaded the 2021, 2022, and 2023 CSV files into DataFrames
- Added a Year column to each dataset
- Reordered columns so that Year is the first column
- Combined the three yearly datasets into a single unified DataFrame
- Performed initial descriptive statistics (df.describe()) to understand data distribution
- Checked dataset shape and number of rows/columns (df.shape)
- Filtered out rows where RBCS_Lvl was 'Unknown' or 'Non-DME'
- Removed unnecessary RUCA-related columns (Suplr_Prvdr_RUCA_Cat, Suplr_Prvdr_RUCA, Suplr_Prvdr_RUCA_Desc)
- Counted null values in each column to identify missing data
- Cleaned string columns:

-- Converted to lowercase

-- Removed special characters

-- Replaced spaces with underscores

-- Filled empty or null strings with 'na'

- Created a new column DME_Sprsn_Ind to flag suppressed rows in Tot_Suplr_Benes
- Reordered columns to place DME_Sprsn_Ind before Tot_Suplr_Benes
- Verified dataset shape and previewed data after cleaning (df.head())
- Saved the cleaned, unified dataset as DMEPOS_cleaned_suplr_serv.csv


End result:

Saved the cleaned dataset to a shared team directory: /dsa/groups/casestudycf25/team02/DMEPOS_cleaned_suplr_serv.csv

## Loading the data and quick exploration

In [1]:
# Loading Libraries
import re
import pandas as pd
from pathlib import Path

Listing all files and folders inside the specified directory

In [2]:
for f in Path("/dsa/groups/casestudycf25/team02/").iterdir():
    print(f.name)

DMEPOS_suplr_2021.csv
Medicare_Monthly_Enrollment_Jun_2025.csv
DMEPOS_suplr_2022.csv
DMEPOS_suplr_2023.csv
DMEPOS_Supplier_Service_2021.csv
DMEPOS_Supplier_Service_2022.csv
DMEPOS_Supplier_Service_2023.csv
LEIE_OIG_Exclusion_List.csv
mup_dme_ry25_p05_v10_dy21_rfrhpr.csv
mup_dme_ry25_p05_v10_dy21_rfrr.csv
mup_dme_ry25_p05_v10_dy22_rfrhpr.csv
mup_dme_ry25_p05_v10_dy22_rfrr.csv
mup_dme_ry25_p05_v10_dy23_rfrhpr.csv
mup_dme_ry25_p05_v10_dy23_rfrr.csv
RUCA2010zipcode.csv
Taxonomy Code List Dec 2023.csv
leie_with_null_npi_clean.csv
leie_with_valid_npi_clean.csv
medicare_enrollment_clean.csv
CMS_General_Payments_2021_2023_raw.csv
casestudycf25t02.sqlite.db
DMEPOS_Bene_only_suplr.csv
DMEPOS_rfrhpr_clean.csv
DMEPOS_rfrr_clean.csv
DMEPOS_RefPro_Ser_clean_02.csv
DMEPOS_cleaned_serv.csv
DMEPOS_cleaned_grouping.csv
OWNRSHP_PGYR2021_2023.csv
ownership_payment_clean.csv
DMEPOS_rfrhpr_clean_labeled.csv
DMEPOS_rfrr_clean_labeled.csv
general_payments_manufacturers_clean.csv
general_payments_providers_cle

In [32]:
#Loading the three yearly DMEPOS Supplier–Service datasets (2021–2023)
df21 = pd.read_csv("/dsa/groups/casestudycf25/team02/DMEPOS_Supplier_Service_2021.csv",na_values=["NA"])

df22 = pd.read_csv("/dsa/groups/casestudycf25/team02/DMEPOS_Supplier_Service_2022.csv",na_values=["NA"])

df23 = pd.read_csv("/dsa/groups/casestudycf25/team02/DMEPOS_Supplier_Service_2023.csv",na_values=["NA"])

# Ensure Suplr_Prvdr_RUCA is treated as numeric
for df_ in [df21, df22, df23]:
    df_["Suplr_Prvdr_RUCA"] = pd.to_numeric(df_["Suplr_Prvdr_RUCA"], errors="coerce")

C:\Users\kamat\AppData\Local\Temp\ipykernel_2220\4010828676.py:2: DtypeWarning: Columns (2,3,4) have mixed types. Specify dtype option on import or set low_memory=False.
  df21 = pd.read_csv(


Taking a quick peak at the dataframes to see if everything is loaded. I can also see the structure of the datasets.

In [33]:
df21.head()

,Suplr_NPI,Suplr_Prvdr_Last_Name_Org,Suplr_Prvdr_First_Name,Suplr_Prvdr_MI,Suplr_Prvdr_Crdntls,Suplr_Prvdr_Ent_Cd,Suplr_Prvdr_St1,Suplr_Prvdr_St2,Suplr_Prvdr_City,Suplr_Prvdr_State_Abrvtn,...,HCPCS_Cd,HCPCS_Desc,Suplr_Rentl_Ind,Tot_Suplr_Benes,Tot_Suplr_Clms,Tot_Suplr_Srvcs,Avg_Suplr_Sbmtd_Chrg,Avg_Suplr_Mdcr_Alowd_Amt,Avg_Suplr_Mdcr_Pymt_Amt,Avg_Suplr_Mdcr_Stdzd_Amt
0,1003000399,"Reconstructive Hand To Shoulder Of Indiana, Llc",NaN,NaN,NaN,O,13431 Old Meridian Street,Suite 225,Carmel,IN,...,L3808,"Wrist hand finger orthosis, rigid without join...",N,69.0,79,82,435.609756,325.202683,257.561707,262.395732
1,1003000399,"Reconstructive Hand To Shoulder Of Indiana, Llc",NaN,NaN,NaN,O,13431 Old Meridian Street,Suite 225,Carmel,IN,...,L3906,"Wrist hand orthosis, without joints, may inclu...",N,30.0,30,35,448.285714,439.852857,351.134286,315.416286
2,1003000399,"Reconstructive Hand To Shoulder Of Indiana, Llc",NaN,NaN,NaN,O,13431 Old Meridian Street,Suite 225,Carmel,IN,...,L3908,"Wrist hand orthosis, wrist extension control c...",N,84.0,99,107,108.457944,66.980000,51.914953,46.384393
3,1003000399,"Reconstructive Hand To Shoulder Of Indiana, Llc",NaN,NaN,NaN,O,13431 Old Meridian Street,Suite 225,Carmel,IN,...,L3913,"Hand finger orthosis, without joints, may incl...",N,16.0,16,16,348.000000,247.410000,197.133125,191.759375
4,1003000399,"Reconstructive Hand To Shoulder Of Indiana, Llc",NaN,NaN,NaN,O,13431 Old Meridian Street,Suite 225,Carmel,IN,...,L3923,"Hand finger orthosis, without joints, may incl...",N,54.0,59,66,113.030303,88.250000,66.411515,65.491667


In [34]:
df22.head()

,Suplr_NPI,Suplr_Prvdr_Last_Name_Org,Suplr_Prvdr_First_Name,Suplr_Prvdr_MI,Suplr_Prvdr_Crdntls,Suplr_Prvdr_Ent_Cd,Suplr_Prvdr_St1,Suplr_Prvdr_St2,Suplr_Prvdr_City,Suplr_Prvdr_State_Abrvtn,...,HCPCS_Cd,HCPCS_Desc,Suplr_Rentl_Ind,Tot_Suplr_Benes,Tot_Suplr_Clms,Tot_Suplr_Srvcs,Avg_Suplr_Sbmtd_Chrg,Avg_Suplr_Mdcr_Alowd_Amt,Avg_Suplr_Mdcr_Pymt_Amt,Avg_Suplr_Mdcr_Stdzd_Amt
0,1003000399,"Reconstructive Hand To Shoulder Of Indiana, Llc",NaN,NaN,NaN,O,13431 Old Meridian Street,Suite 225,Carmel,IN,...,L3808,"Wrist hand finger orthosis, rigid without join...",N,62.0,62,65,489.230769,342.030000,269.987077,278.113692
1,1003000399,"Reconstructive Hand To Shoulder Of Indiana, Llc",NaN,NaN,NaN,O,13431 Old Meridian Street,Suite 225,Carmel,IN,...,L3906,"Wrist hand orthosis, without joints, may inclu...",N,31.0,31,32,503.562500,467.656875,363.158125,326.565313
2,1003000399,"Reconstructive Hand To Shoulder Of Indiana, Llc",NaN,NaN,NaN,O,13431 Old Meridian Street,Suite 225,Carmel,IN,...,L3908,"Wrist hand orthosis, wrist extension control c...",N,61.0,61,67,110.447761,70.400000,53.040746,48.040299
3,1003000399,"Reconstructive Hand To Shoulder Of Indiana, Llc",NaN,NaN,NaN,O,13431 Old Meridian Street,Suite 225,Carmel,IN,...,L3913,"Hand finger orthosis, without joints, may incl...",N,12.0,12,12,376.000000,260.030000,205.073333,202.370000
4,1003000399,"Reconstructive Hand To Shoulder Of Indiana, Llc",NaN,NaN,NaN,O,13431 Old Meridian Street,Suite 225,Carmel,IN,...,L3923,"Hand finger orthosis, without joints, may incl...",N,30.0,30,32,115.000000,92.750000,69.985313,69.994062


In [35]:
df23.head()

,Suplr_NPI,Suplr_Prvdr_Last_Name_Org,Suplr_Prvdr_First_Name,Suplr_Prvdr_MI,Suplr_Prvdr_Crdntls,Suplr_Prvdr_Ent_Cd,Suplr_Prvdr_St1,Suplr_Prvdr_St2,Suplr_Prvdr_City,Suplr_Prvdr_State_Abrvtn,...,HCPCS_Cd,HCPCS_Desc,Suplr_Rentl_Ind,Tot_Suplr_Benes,Tot_Suplr_Clms,Tot_Suplr_Srvcs,Avg_Suplr_Sbmtd_Chrg,Avg_Suplr_Mdcr_Alowd_Amt,Avg_Suplr_Mdcr_Pymt_Amt,Avg_Suplr_Mdcr_Stdzd_Amt
0,1003000399,"Reconstructive Hand To Shoulder Of Indiana, Llc",NaN,NaN,NaN,O,13431 Old Meridian Street,Suite 225,Carmel,IN,...,L3808,"Wrist hand finger orthosis, rigid without join...",N,38.0,39,39,540.000000,371.79,289.320256,300.242820
1,1003000399,"Reconstructive Hand To Shoulder Of Indiana, Llc",NaN,NaN,NaN,O,13431 Old Meridian Street,Suite 225,Carmel,IN,...,L3906,"Wrist hand orthosis, without joints, may inclu...",N,27.0,27,28,555.178571,528.91,414.603214,361.113214
2,1003000399,"Reconstructive Hand To Shoulder Of Indiana, Llc",NaN,NaN,NaN,O,13431 Old Meridian Street,Suite 225,Carmel,IN,...,L3908,"Wrist hand orthosis, wrist extension control c...",N,57.0,57,63,112.619048,76.52,59.047619,53.890794
3,1003000399,"Reconstructive Hand To Shoulder Of Indiana, Llc",NaN,NaN,NaN,O,13431 Old Meridian Street,Suite 225,Carmel,IN,...,L3923,"Hand finger orthosis, without joints, may incl...",N,24.0,24,27,115.000000,100.82,78.927778,79.388148
4,1003000399,"Reconstructive Hand To Shoulder Of Indiana, Llc",NaN,NaN,NaN,O,13431 Old Meridian Street,Suite 225,Carmel,IN,...,L3925,"Finger orthosis, proximal interphalangeal (pip...",N,13.0,13,13,99.769231,68.56,42.296154,34.993077


In [36]:
# Looking at the column names....

df23.columns.tolist()

['Suplr_NPI',
 'Suplr_Prvdr_Last_Name_Org',
 'Suplr_Prvdr_First_Name',
 'Suplr_Prvdr_MI',
 'Suplr_Prvdr_Crdntls',
 'Suplr_Prvdr_Ent_Cd',
 'Suplr_Prvdr_St1',
 'Suplr_Prvdr_St2',
 'Suplr_Prvdr_City',
 'Suplr_Prvdr_State_Abrvtn',
 'Suplr_Prvdr_State_FIPS',
 'Suplr_Prvdr_Zip5',
 'Suplr_Prvdr_RUCA_Cat',
 'Suplr_Prvdr_RUCA',
 'Suplr_Prvdr_RUCA_Desc',
 'Suplr_Prvdr_Cntry',
 'Suplr_Prvdr_Spclty_Cd',
 'Suplr_Prvdr_Spclty_Desc',
 'Suplr_Prvdr_Spclty_Srce',
 'RBCS_Lvl',
 'RBCS_Id',
 'RBCS_Desc',
 'HCPCS_Cd',
 'HCPCS_Desc',
 'Suplr_Rentl_Ind',
 'Tot_Suplr_Benes',
 'Tot_Suplr_Clms',
 'Tot_Suplr_Srvcs',
 'Avg_Suplr_Sbmtd_Chrg',
 'Avg_Suplr_Mdcr_Alowd_Amt',
 'Avg_Suplr_Mdcr_Pymt_Amt',
 'Avg_Suplr_Mdcr_Stdzd_Amt']

## Carpentry

The below code adds a Year column to each yearly dataset, reorders columns so that Year comes first, and then combines all three years into a single DataFrame. The result is one dataset with a consistent structure across 2021, 2022, and 2023, that we can use for analysis.

In [37]:
# Adding a year column
df21["Year"] = 2021
df22["Year"] = 2022
df23["Year"] = 2023

# Reordering columns
df21 = df21[["Year"] + [c for c in df21.columns if c != "Year"]]
df22 = df22[["Year"] + [c for c in df22.columns if c != "Year"]]
df23 = df23[["Year"] + [c for c in df23.columns if c != "Year"]]

# Combining all three datasets
df = pd.concat([df21, df22, df23], ignore_index=True)

In [38]:
# Checking...
df.head()

,Year,Suplr_NPI,Suplr_Prvdr_Last_Name_Org,Suplr_Prvdr_First_Name,Suplr_Prvdr_MI,Suplr_Prvdr_Crdntls,Suplr_Prvdr_Ent_Cd,Suplr_Prvdr_St1,Suplr_Prvdr_St2,Suplr_Prvdr_City,...,HCPCS_Cd,HCPCS_Desc,Suplr_Rentl_Ind,Tot_Suplr_Benes,Tot_Suplr_Clms,Tot_Suplr_Srvcs,Avg_Suplr_Sbmtd_Chrg,Avg_Suplr_Mdcr_Alowd_Amt,Avg_Suplr_Mdcr_Pymt_Amt,Avg_Suplr_Mdcr_Stdzd_Amt
0,2021,1003000399,"Reconstructive Hand To Shoulder Of Indiana, Llc",NaN,NaN,NaN,O,13431 Old Meridian Street,Suite 225,Carmel,...,L3808,"Wrist hand finger orthosis, rigid without join...",N,69.0,79,82,435.609756,325.202683,257.561707,262.395732
1,2021,1003000399,"Reconstructive Hand To Shoulder Of Indiana, Llc",NaN,NaN,NaN,O,13431 Old Meridian Street,Suite 225,Carmel,...,L3906,"Wrist hand orthosis, without joints, may inclu...",N,30.0,30,35,448.285714,439.852857,351.134286,315.416286
2,2021,1003000399,"Reconstructive Hand To Shoulder Of Indiana, Llc",NaN,NaN,NaN,O,13431 Old Meridian Street,Suite 225,Carmel,...,L3908,"Wrist hand orthosis, wrist extension control c...",N,84.0,99,107,108.457944,66.980000,51.914953,46.384393
3,2021,1003000399,"Reconstructive Hand To Shoulder Of Indiana, Llc",NaN,NaN,NaN,O,13431 Old Meridian Street,Suite 225,Carmel,...,L3913,"Hand finger orthosis, without joints, may incl...",N,16.0,16,16,348.000000,247.410000,197.133125,191.759375
4,2021,1003000399,"Reconstructive Hand To Shoulder Of Indiana, Llc",NaN,NaN,NaN,O,13431 Old Meridian Street,Suite 225,Carmel,...,L3923,"Hand finger orthosis, without joints, may incl...",N,54.0,59,66,113.030303,88.250000,66.411515,65.491667


In [39]:
# Checking...
df.tail()

,Year,Suplr_NPI,Suplr_Prvdr_Last_Name_Org,Suplr_Prvdr_First_Name,Suplr_Prvdr_MI,Suplr_Prvdr_Crdntls,Suplr_Prvdr_Ent_Cd,Suplr_Prvdr_St1,Suplr_Prvdr_St2,Suplr_Prvdr_City,...,HCPCS_Cd,HCPCS_Desc,Suplr_Rentl_Ind,Tot_Suplr_Benes,Tot_Suplr_Clms,Tot_Suplr_Srvcs,Avg_Suplr_Sbmtd_Chrg,Avg_Suplr_Mdcr_Alowd_Amt,Avg_Suplr_Mdcr_Pymt_Amt,Avg_Suplr_Mdcr_Stdzd_Amt
1454469,2023,1992989909,Joseph P Gabryszewski,NaN,NaN,NaN,O,3 Kirchner Ave,NaN,Hyde Park,...,A5513,"For diabetics only, multiple density insert, c...",N,29.0,29,172,56.62,51.47,40.353372,40.350000
1454470,2023,1992999106,"Northwest Indiana Eye Associates, Pc",NaN,NaN,NaN,O,297 W. Franciscan Dr.,Suite 101,Crown Point,...,V2020,"Frames, purchases",N,72.0,89,89,70.00,70.00,53.646742,61.208989
1454471,2023,1992999106,"Northwest Indiana Eye Associates, Pc",NaN,NaN,NaN,O,297 W. Franciscan Dr.,Suite 101,Crown Point,...,V2103,"Spherocylinder, single vision, plano to plus o...",N,24.0,29,58,45.00,44.71,35.050000,33.850000
1454472,2023,1992999106,"Northwest Indiana Eye Associates, Pc",NaN,NaN,NaN,O,297 W. Franciscan Dr.,Suite 101,Crown Point,...,V2203,"Spherocylinder, bifocal, plano to plus or minu...",N,20.0,28,54,65.00,61.60,48.290000,51.480000
1454473,2023,1992999106,"Northwest Indiana Eye Associates, Pc",NaN,NaN,NaN,O,297 W. Franciscan Dr.,Suite 101,Crown Point,...,V2303,"Spherocylinder, trifocal, plano to plus or min...",N,28.0,30,60,70.00,70.00,46.640667,55.972333


In [40]:
# Looking into descriptive stats
df.describe()

,Year,Suplr_NPI,Suplr_Prvdr_State_FIPS,Suplr_Prvdr_Zip5,Suplr_Prvdr_RUCA,Tot_Suplr_Benes,Tot_Suplr_Clms,Tot_Suplr_Srvcs,Avg_Suplr_Sbmtd_Chrg,Avg_Suplr_Mdcr_Alowd_Amt,Avg_Suplr_Mdcr_Pymt_Amt,Avg_Suplr_Mdcr_Stdzd_Amt
count,1.454474e+06,1.454474e+06,1.454474e+06,1.454474e+06,1.454380e+06,864704.000000,1.454474e+06,1.454474e+06,1.454474e+06,1.454474e+06,1.454474e+06,1.454474e+06
mean,2.021970e+03,1.502034e+09,2.856412e+01,4.914111e+04,2.134648e+00,97.770228,1.647799e+02,4.576480e+03,2.056878e+02,1.010626e+02,7.798224e+01,7.767823e+01
std,8.168502e-01,2.883286e+08,1.550741e+01,2.749516e+04,2.640066e+00,620.189075,1.892079e+03,4.952015e+05,9.028967e+02,4.198971e+02,3.283998e+02,3.193373e+02
min,2.021000e+03,1.003000e+09,1.000000e+00,6.050000e+02,1.000000e+00,11.000000,1.100000e+01,1.100000e+01,3.348654e-04,3.348654e-04,0.000000e+00,0.000000e+00
25%,2.021000e+03,1.255406e+09,1.700000e+01,2.836500e+04,1.000000e+00,16.000000,1.600000e+01,2.700000e+01,1.321108e+01,2.799809e+00,2.087560e+00,2.183343e+00
50%,2.022000e+03,1.508413e+09,2.800000e+01,4.656300e+04,1.000000e+00,28.000000,2.900000e+01,9.600000e+01,3.659725e+01,1.600000e+01,1.261647e+01,1.254000e+01
75%,2.023000e+03,1.750331e+09,4.200000e+01,7.274500e+04,2.000000e+00,65.000000,7.900000e+01,6.960000e+02,1.150000e+02,5.661000e+01,4.196140e+01,4.371000e+01
max,2.023000e+03,1.992999e+09,7.800000e+01,9.990100e+04,9.900000e+01,112797.000000,8.590970e+05,2.967321e+08,7.761210e+04,2.773326e+04,2.174288e+04,2.149642e+04


In [41]:
# Data types
df.dtypes

Year                           int64
Suplr_NPI                      int64
Suplr_Prvdr_Last_Name_Org     object
Suplr_Prvdr_First_Name        object
Suplr_Prvdr_MI                object
Suplr_Prvdr_Crdntls           object
Suplr_Prvdr_Ent_Cd            object
Suplr_Prvdr_St1               object
Suplr_Prvdr_St2               object
Suplr_Prvdr_City              object
Suplr_Prvdr_State_Abrvtn      object
Suplr_Prvdr_State_FIPS         int64
Suplr_Prvdr_Zip5               int64
Suplr_Prvdr_RUCA_Cat          object
Suplr_Prvdr_RUCA             float64
Suplr_Prvdr_RUCA_Desc         object
Suplr_Prvdr_Cntry             object
Suplr_Prvdr_Spclty_Cd         object
Suplr_Prvdr_Spclty_Desc       object
Suplr_Prvdr_Spclty_Srce       object
RBCS_Lvl                      object
RBCS_Id                       object
RBCS_Desc                     object
HCPCS_Cd                      object
HCPCS_Desc                    object
Suplr_Rentl_Ind               object
Tot_Suplr_Benes              float64
T

In [42]:
# Shape
df.shape

(1454474, 33)

There are some entries that don't really represent DME, they will be removed for time being

In [43]:
# Removing entries where 'RBCS_Lvl' is 'Unknown' or 'Non-DME'
RBCS_lvl_ = ['Unknown', 'Non-DME']
df_clean = df[~df['RBCS_Lvl'].isin(RBCS_lvl_)]

# Checking
df_clean.head()

,Year,Suplr_NPI,Suplr_Prvdr_Last_Name_Org,Suplr_Prvdr_First_Name,Suplr_Prvdr_MI,Suplr_Prvdr_Crdntls,Suplr_Prvdr_Ent_Cd,Suplr_Prvdr_St1,Suplr_Prvdr_St2,Suplr_Prvdr_City,...,HCPCS_Cd,HCPCS_Desc,Suplr_Rentl_Ind,Tot_Suplr_Benes,Tot_Suplr_Clms,Tot_Suplr_Srvcs,Avg_Suplr_Sbmtd_Chrg,Avg_Suplr_Mdcr_Alowd_Amt,Avg_Suplr_Mdcr_Pymt_Amt,Avg_Suplr_Mdcr_Stdzd_Amt
0,2021,1003000399,"Reconstructive Hand To Shoulder Of Indiana, Llc",NaN,NaN,NaN,O,13431 Old Meridian Street,Suite 225,Carmel,...,L3808,"Wrist hand finger orthosis, rigid without join...",N,69.0,79,82,435.609756,325.202683,257.561707,262.395732
1,2021,1003000399,"Reconstructive Hand To Shoulder Of Indiana, Llc",NaN,NaN,NaN,O,13431 Old Meridian Street,Suite 225,Carmel,...,L3906,"Wrist hand orthosis, without joints, may inclu...",N,30.0,30,35,448.285714,439.852857,351.134286,315.416286
2,2021,1003000399,"Reconstructive Hand To Shoulder Of Indiana, Llc",NaN,NaN,NaN,O,13431 Old Meridian Street,Suite 225,Carmel,...,L3908,"Wrist hand orthosis, wrist extension control c...",N,84.0,99,107,108.457944,66.980000,51.914953,46.384393
3,2021,1003000399,"Reconstructive Hand To Shoulder Of Indiana, Llc",NaN,NaN,NaN,O,13431 Old Meridian Street,Suite 225,Carmel,...,L3913,"Hand finger orthosis, without joints, may incl...",N,16.0,16,16,348.000000,247.410000,197.133125,191.759375
4,2021,1003000399,"Reconstructive Hand To Shoulder Of Indiana, Llc",NaN,NaN,NaN,O,13431 Old Meridian Street,Suite 225,Carmel,...,L3923,"Hand finger orthosis, without joints, may incl...",N,54.0,59,66,113.030303,88.250000,66.411515,65.491667


In [44]:
df_clean.shape
#Around 200k entries were removed

(1218152, 33)

I will be removing RUCA-related attributes for the time now.
- Suplr_Prvdr_RUCA_Cat
- Suplr_Prvdr_RUCA
- Suplr_Prvdr_RUCA_Desc

In [45]:
# Removing RUCA-related attributes
rem_att = ['Suplr_Prvdr_RUCA_Cat', 'Suplr_Prvdr_RUCA', 'Suplr_Prvdr_RUCA_Desc']
df = df_clean.drop(columns=rem_att)

# Checking ..
df.head()

,Year,Suplr_NPI,Suplr_Prvdr_Last_Name_Org,Suplr_Prvdr_First_Name,Suplr_Prvdr_MI,Suplr_Prvdr_Crdntls,Suplr_Prvdr_Ent_Cd,Suplr_Prvdr_St1,Suplr_Prvdr_St2,Suplr_Prvdr_City,...,HCPCS_Cd,HCPCS_Desc,Suplr_Rentl_Ind,Tot_Suplr_Benes,Tot_Suplr_Clms,Tot_Suplr_Srvcs,Avg_Suplr_Sbmtd_Chrg,Avg_Suplr_Mdcr_Alowd_Amt,Avg_Suplr_Mdcr_Pymt_Amt,Avg_Suplr_Mdcr_Stdzd_Amt
0,2021,1003000399,"Reconstructive Hand To Shoulder Of Indiana, Llc",NaN,NaN,NaN,O,13431 Old Meridian Street,Suite 225,Carmel,...,L3808,"Wrist hand finger orthosis, rigid without join...",N,69.0,79,82,435.609756,325.202683,257.561707,262.395732
1,2021,1003000399,"Reconstructive Hand To Shoulder Of Indiana, Llc",NaN,NaN,NaN,O,13431 Old Meridian Street,Suite 225,Carmel,...,L3906,"Wrist hand orthosis, without joints, may inclu...",N,30.0,30,35,448.285714,439.852857,351.134286,315.416286
2,2021,1003000399,"Reconstructive Hand To Shoulder Of Indiana, Llc",NaN,NaN,NaN,O,13431 Old Meridian Street,Suite 225,Carmel,...,L3908,"Wrist hand orthosis, wrist extension control c...",N,84.0,99,107,108.457944,66.980000,51.914953,46.384393
3,2021,1003000399,"Reconstructive Hand To Shoulder Of Indiana, Llc",NaN,NaN,NaN,O,13431 Old Meridian Street,Suite 225,Carmel,...,L3913,"Hand finger orthosis, without joints, may incl...",N,16.0,16,16,348.000000,247.410000,197.133125,191.759375
4,2021,1003000399,"Reconstructive Hand To Shoulder Of Indiana, Llc",NaN,NaN,NaN,O,13431 Old Meridian Street,Suite 225,Carmel,...,L3923,"Hand finger orthosis, without joints, may incl...",N,54.0,59,66,113.030303,88.250000,66.411515,65.491667


Now to check for empty values in the dataset.

In [46]:
# Count nulls for each attribute
nulls = df.isnull().sum()

# Print null counts
for i in nulls:
    print(i)

0
0
0
1214948
1215847
1215667
0
0
1030954
0
0
0
0
0
0
0
0
0
0
0
0
0
0
403652
0
0
0
0
0
0


The first four are string values related to supplier information, such as middle and last name, 2nd street address. Only 1 numerical value has empty values due to suppression.

In [47]:
# Finding string columns
string_cols = df.select_dtypes(include='object').columns.tolist()

# Cleaning string columns
for c in string_cols:
    df[c] = df[c].str.lower()  # convert to lowercase
    df[c] = df[c].str.replace(r"[^A-Za-z0-9\s]", "", regex=True)  # remove special characters
    df[c] = df[c].str.replace(r"\s+", "_", regex=True)  # replace spaces with underscores
    df[c] = df[c].replace("", "na").fillna("na")  # replace empty or null with 'na'

In [48]:
# Checking..
df.shape

(1218152, 30)

In [49]:
# Checking
df.head()

,Year,Suplr_NPI,Suplr_Prvdr_Last_Name_Org,Suplr_Prvdr_First_Name,Suplr_Prvdr_MI,Suplr_Prvdr_Crdntls,Suplr_Prvdr_Ent_Cd,Suplr_Prvdr_St1,Suplr_Prvdr_St2,Suplr_Prvdr_City,...,HCPCS_Cd,HCPCS_Desc,Suplr_Rentl_Ind,Tot_Suplr_Benes,Tot_Suplr_Clms,Tot_Suplr_Srvcs,Avg_Suplr_Sbmtd_Chrg,Avg_Suplr_Mdcr_Alowd_Amt,Avg_Suplr_Mdcr_Pymt_Amt,Avg_Suplr_Mdcr_Stdzd_Amt
0,2021,1003000399,reconstructive_hand_to_shoulder_of_indiana_llc,na,na,na,o,13431_old_meridian_street,suite_225,carmel,...,l3808,wrist_hand_finger_orthosis_rigid_without_joint...,n,69.0,79,82,435.609756,325.202683,257.561707,262.395732
1,2021,1003000399,reconstructive_hand_to_shoulder_of_indiana_llc,na,na,na,o,13431_old_meridian_street,suite_225,carmel,...,l3906,wrist_hand_orthosis_without_joints_may_include...,n,30.0,30,35,448.285714,439.852857,351.134286,315.416286
2,2021,1003000399,reconstructive_hand_to_shoulder_of_indiana_llc,na,na,na,o,13431_old_meridian_street,suite_225,carmel,...,l3908,wrist_hand_orthosis_wrist_extension_control_co...,n,84.0,99,107,108.457944,66.980000,51.914953,46.384393
3,2021,1003000399,reconstructive_hand_to_shoulder_of_indiana_llc,na,na,na,o,13431_old_meridian_street,suite_225,carmel,...,l3913,hand_finger_orthosis_without_joints_may_includ...,n,16.0,16,16,348.000000,247.410000,197.133125,191.759375
4,2021,1003000399,reconstructive_hand_to_shoulder_of_indiana_llc,na,na,na,o,13431_old_meridian_street,suite_225,carmel,...,l3923,hand_finger_orthosis_without_joints_may_includ...,n,54.0,59,66,113.030303,88.250000,66.411515,65.491667


In [50]:
# Create a column to indicate suppressed rows
df["DME_Sprsn_Ind"] = df["Tot_Suplr_Benes"].isnull().apply(lambda x: "y" if x else "n")

In [51]:
# Reordering the columns
cols = df.columns.tolist()
insert_at = cols.index("Tot_Suplr_Benes")

new_order = cols[:insert_at] + ["DME_Sprsn_Ind"] + cols[insert_at:]
df = df[new_order]

In [52]:
# Checking..
df.head()

,Year,Suplr_NPI,Suplr_Prvdr_Last_Name_Org,Suplr_Prvdr_First_Name,Suplr_Prvdr_MI,Suplr_Prvdr_Crdntls,Suplr_Prvdr_Ent_Cd,Suplr_Prvdr_St1,Suplr_Prvdr_St2,Suplr_Prvdr_City,...,Suplr_Rentl_Ind,DME_Sprsn_Ind,Tot_Suplr_Benes,Tot_Suplr_Clms,Tot_Suplr_Srvcs,Avg_Suplr_Sbmtd_Chrg,Avg_Suplr_Mdcr_Alowd_Amt,Avg_Suplr_Mdcr_Pymt_Amt,Avg_Suplr_Mdcr_Stdzd_Amt,DME_Sprsn_Ind
0,2021,1003000399,reconstructive_hand_to_shoulder_of_indiana_llc,na,na,na,o,13431_old_meridian_street,suite_225,carmel,...,n,n,69.0,79,82,435.609756,325.202683,257.561707,262.395732,n
1,2021,1003000399,reconstructive_hand_to_shoulder_of_indiana_llc,na,na,na,o,13431_old_meridian_street,suite_225,carmel,...,n,n,30.0,30,35,448.285714,439.852857,351.134286,315.416286,n
2,2021,1003000399,reconstructive_hand_to_shoulder_of_indiana_llc,na,na,na,o,13431_old_meridian_street,suite_225,carmel,...,n,n,84.0,99,107,108.457944,66.980000,51.914953,46.384393,n
3,2021,1003000399,reconstructive_hand_to_shoulder_of_indiana_llc,na,na,na,o,13431_old_meridian_street,suite_225,carmel,...,n,n,16.0,16,16,348.000000,247.410000,197.133125,191.759375,n
4,2021,1003000399,reconstructive_hand_to_shoulder_of_indiana_llc,na,na,na,o,13431_old_meridian_street,suite_225,carmel,...,n,n,54.0,59,66,113.030303,88.250000,66.411515,65.491667,n


In [53]:
# Checking..
df.shape

(1218152, 32)

In [54]:
df.to_csv("/dsa/groups/casestudycf25/team02/DMEPOS_cleaned_suplr_serv.csv", index=False)